# Big Five CopDDE + CSP analysis with `semidde`

This notebook is a compact example of applying the `semidde` package to the YouGov Big Five data used in the CopDDE paper. It mirrors the original real-data workflow as closely as possible:

- read `Data/WASU0003_B5_OUTPUT.sav`;
- keep the 98 Big Five items used by the paper;
- build rank-likelihood inputs before median-imputing missing item responses;
- use a two-layer copula DDE model with CSP dimension selection;
- select hyperparameters with predictive pMSE and `refit=True`.

For a quick demonstration, the notebook defaults to a small smoke run. Set `RUN_FULL_GRID = True` to run the paper-style grid: 192 candidates, 50 pMSE synthetic repetitions, `maxiter=50`, and `convergence_std=0.01`.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "scripts":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "SemiDDE":
    REPO_ROOT = REPO_ROOT.parent

SEMIDDE_ROOT = REPO_ROOT / "SemiDDE"
SRC = SEMIDDE_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(SEMIDDE_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(SEMIDDE_ROOT / "scripts"))

from semidde import select_hyperparameters
from semidde.algorithms.copula import normal_initialize_copula_dde
from semidde.copula import simulate_gaussian_mixture

from realdata_b5_copula_csp import (
    DEFAULT_SAV,
    PAPER_TAUS,
    PAPER_T_SPIKES,
    PAPER_TEMPS,
    anchor_tables,
    active_latent_frame,
    ideology_tree_summary,
    load_b5_data,
    paper_candidates,
    selection_summary,
    write_outputs,
)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)

## 1. Analysis settings

The original-paper setting is shown below. The smoke run keeps the same code path but uses only a tiny subset of the grid, one SAEM iteration, and one pMSE repetition so that the notebook can be executed interactively.

In [ ]:
RUN_FULL_GRID = False  # Set True for the paper-style run.
RANDOM_STATE = 20260712

sav_path = REPO_ROOT / "Data" / "WASU0003_B5_OUTPUT.sav"
out_dir = REPO_ROOT / "Results" / (
    "notebook_b5_copula_csp_full" if RUN_FULL_GRID else "notebook_b5_copula_csp_smoke"
)

optimizer_maxiter = 50
convergence_std = 0.01
pmse_n_estimators = 100 if RUN_FULL_GRID else 20
n_repeats = 50 if RUN_FULL_GRID else 1

candidates = paper_candidates(
    smoke=not RUN_FULL_GRID,
    optimizer_maxiter=optimizer_maxiter,
    convergence_std=convergence_std,
)

print(f"Data file: {sav_path}")
print(f"Output directory: {out_dir}")
print(f"Candidates: {len(candidates)} | pMSE repeats: {n_repeats} | RF trees: {pmse_n_estimators}")
print({"taus": PAPER_TAUS, "t_spikes": PAPER_T_SPIKES, "temps": PAPER_TEMPS})

## 2. Load and preprocess the Big Five data

The helper below follows the original MATLAB script: raw columns 3:102 are used, raw columns 46 and 53 are dropped, ranks are computed before replacing missing item responses by 3, and the two CSP upper bounds are `floor(J/3)` and `floor(K1/3)`.

In [ ]:
data = load_b5_data(sav_path)

print(f"N = {data.X_filled.shape[0]}")
print(f"J = {data.X_filled.shape[1]}")
print(f"K_list = {data.K_list}")
print(f"Missing selected cells = {int(np.isnan(data.X_rank_source).sum())}")
print(f"Response levels = {sorted(set(data.Rlevels.tolist()))}")

pd.DataFrame({
    "variable": data.selected_names[:8],
    "label": [data.selected_labels[name] for name in data.selected_names[:8]],
})

## 3. Shared initialization

To make candidates comparable, the workflow precomputes the copula latent `Z` and the normal initialization once, then passes the same initialization to every hyperparameter candidate.

In [ ]:
Z_init = simulate_gaussian_mixture(
    data.X_filled,
    data.X_filled.shape[1],
    random_state=RANDOM_STATE,
)

init = normal_initialize_copula_dde(
    data.X_filled,
    data.K_list,
    Z=Z_init,
    K1_max=data.K_list[0],
    random_state=RANDOM_STATE,
    epsilon=1e-5,
)

fixed_hp = {
    "K_list": data.K_list,
    "Z": Z_init,
    "R": data.R,
    "Rlevels": data.Rlevels,
    "prop_in": init["prop"],
    "gamma_in": init["gamma"],
    "B_list_in": init["B_list"],
    "A_list_in": init["A_list"],
}

print("Shared initialization ready.")

## 4. Select hyperparameters and refit the best model

This is the central `semidde` call. `select_hyperparameters(..., refit=True)` first scores all candidates, then refits and returns the best model object.

In [ ]:
result = select_hyperparameters(
    data.X_filled,
    D=2,
    model="copula",
    dimension_method="csp",
    criterion="pmse",
    candidates=candidates,
    fixed_hp=fixed_hp,
    n_repeats=n_repeats,
    pmse_n_estimators=pmse_n_estimators,
    refit=True,
    random_state=RANDOM_STATE,
)

summary = selection_summary(result)
summary

## 5. Inspect model-selection results

The CV table contains one row per candidate. Lower `mean_pmse` is better.

In [ ]:
cv_table = pd.DataFrame(result.diagnostics["cv_table"])
display_cols = [
    "candidate_index",
    "name",
    "tau",
    "t_spike",
    "temp",
    "mean_pmse",
    "std_pmse",
    "fit_seconds",
    "score_seconds",
]

cv_table.sort_values("mean_pmse").loc[:, [c for c in display_cols if c in cv_table.columns]].head(10)

In [ ]:
B_active = [int(np.any(np.asarray(B)[:, 1:] != 0, axis=0).sum()) for B in result.B_list]
print(f"Active latent dimensions by layer: {B_active}")
print("Paper reports K* = (9, 1).")

## 6. Summarize the fitted parameters

For interpretability, we extract anchor-like items for each active shallow latent dimension. Because latent columns are exchangeable, compare these themes with the paper by content rather than by column number.

In [ ]:
b1_anchors, b2_anchors = anchor_tables(result, data)

b1_anchors.query("rank <= 3").loc[:, [
    "latent_index",
    "rank",
    "item",
    "label",
    "anchor_score",
    "loading",
]].head(30)

In [ ]:
b2_anchors.loc[:, ["latent_index", "rank", "item", "anchor_score", "loading"]].head(20)

## 7. External validity check: ideology groups

The paper compares very liberal and very conservative respondents. The code below computes posterior latent means and summarizes the same two ideology endpoints.

In [ ]:
latent = active_latent_frame(result, data.raw)
latent_cols = [c for c in latent.columns if c.startswith("A1_") or c.startswith("A2_")]

ideology_means = (
    latent.loc[latent["ideo5"].isin([1, 5])]
    .groupby("ideo5")[latent_cols]
    .mean()
    .T
    .rename(columns={1: "very_liberal", 5: "very_conservative"})
)
ideology_means["difference_conservative_minus_liberal"] = (
    ideology_means["very_conservative"] - ideology_means["very_liberal"]
)
ideology_means.sort_values("difference_conservative_minus_liberal", key=lambda s: s.abs(), ascending=False)

In [ ]:
tree_pred, tree_summary = ideology_tree_summary(latent, random_state=RANDOM_STATE)
tree_summary

## 8. Save outputs

The output writer saves the CV table, timing table, anchor summaries, posterior latent means, the ideology tree summary, and the fitted result pickle.

In [ ]:
config = {
    "sav_path": sav_path,
    "out_dir": out_dir,
    "run_full_grid": RUN_FULL_GRID,
    "n_candidates": len(candidates),
    "n_repeats": n_repeats,
    "pmse_n_estimators": pmse_n_estimators,
    "random_state": RANDOM_STATE,
    "optimizer_maxiter": optimizer_maxiter,
    "convergence_std": convergence_std,
    "epsilon": 1e-5,
    "shared_initialization": True,
    "paper_grid": {
        "taus": PAPER_TAUS,
        "t_spikes": PAPER_T_SPIKES,
        "temps": PAPER_TEMPS,
    },
}

write_outputs(out_dir=out_dir, data=data, result=result, config=config)
print(f"Saved outputs to: {out_dir}")

## 9. Notes for comparing with the CopDDE paper

- The paper reports a two-layer CopDDE-CSP model with `K*=(9,1)`.
- The fitted latent columns are exchangeable, so Table 3 should be compared by key-item themes rather than by exact latent index.
- The binary latent labels and loading signs are not identified; a Figure 6-style ideology comparison should focus on separation magnitude and theme, not the literal 0/1 direction.
- For the full reproduction run, use `RUN_FULL_GRID = True`. The recent local full run completed in roughly 5.5 hours on this machine.